# 12 - Documentation RAG Backend with Gemini and Pinecone

## Objective

Reproduce the role of the instructor's `backend/core.py` using the Gemini-compatible index created in notebooks 10 and 11.

```text
User question + optional chat history
        ↓
History-aware query reformulation
        ↓
Gemini query embedding, 1,536 dimensions
        ↓
Pinecone retrieval from namespace langchain-docs
        ↓
Grounded answer prompt
        ↓
Gemini chat model
        ↓
{
    "answer": str,
    "context": list[Document],
    "sources": list[str],
    "standalone_question": str
}
```

## Instructor logic preserved

- Reusable backend initialization
- Retriever over the documentation vector store
- Conversational history for follow-up questions
- Context-aware question reformulation
- Retrieval-augmented answer generation
- Returned context documents for source citations
- Reusable `run_llm()` function for the Streamlit frontend

## Gemini adaptations

- `ChatGoogleGenerativeAI` replaces `ChatOpenAI`.
- `GoogleGenerativeAIEmbeddings` replaces `OpenAIEmbeddings`.
- Retrieval uses the same `gemini-embedding-001` model and 1,536 dimensions used during ingestion.
- The Pinecone namespace is explicitly set to `langchain-docs`.

> Complete notebook 11 successfully before running retrieval tests here.


## 1. Optional package installation

Run only if an import fails, then restart the kernel.


In [ ]:
# %pip install -U \
#     langchain \
#     langchain-core \
#     langchain-google-genai \
#     langchain-pinecone \
#     pinecone \
#     python-dotenv \
#     truststore


## 2. Verify the active Python kernel

In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 3. Configure corporate SSL

Run this before importing clients that make HTTPS requests.


In [2]:
import truststore

truststore.inject_into_ssl()
print("Windows trust store injected into Python SSL.")


Windows trust store injected into Python SSL.


## 4. Import backend components

In [7]:
import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain_classic.chains import create_history_aware_retriever
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

print("Imports completed successfully.")


Imports completed successfully.


## 5. Notebook-friendly logging helpers

These retain the supporting role of the instructor's `logger.py`.


In [9]:
class Colors:
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    PURPLE = "\033[95m"
    BOLD = "\033[1m"
    END = "\033[0m"


def log_info(message):
    print(f"{Colors.CYAN}INFO: {message}{Colors.END}")


def log_success(message):
    print(f"{Colors.GREEN}SUCCESS: {message}{Colors.END}")


def log_warning(message):
    print(f"{Colors.YELLOW}WARNING: {message}{Colors.END}")


def log_error(message):
    print(f"{Colors.RED}ERROR: {message}{Colors.END}")


def log_header(message):
    print()
    print(Colors.BOLD + Colors.PURPLE + "=" * 72 + Colors.END)
    print(Colors.BOLD + Colors.PURPLE + message + Colors.END)
    print(Colors.BOLD + Colors.PURPLE + "=" * 72 + Colors.END)
    print()


log_success("Logging helpers are ready.")


SUCCESS: Logging helpers are ready.


## 6. Locate and load `.env`

Expected variables:

```env
GOOGLE_API_KEY=your-google-key
PINECONE_API_KEY=your-pinecone-key
DOCUMENTATION_INDEX_NAME=langchain-docs-helper-gemini
```

Optional:

```env
GEMINI_CHAT_MODEL=gemini-3.5-flash-lite
```


In [10]:
CURRENT_DIR = Path.cwd().resolve()

env_candidates = [
    CURRENT_DIR / ".env",
    CURRENT_DIR.parent / "langchain-course" / ".env",
]

ENV_PATH = next(
    (candidate for candidate in env_candidates if candidate.exists()),
    None,
)

if ENV_PATH is None:
    raise FileNotFoundError(
        "Could not find .env in the notebook folder or sibling langchain-course folder."
    )

load_dotenv(dotenv_path=ENV_PATH)

required_variables = [
    "GOOGLE_API_KEY",
    "PINECONE_API_KEY",
    "DOCUMENTATION_INDEX_NAME",
]

missing_variables = [
    variable_name
    for variable_name in required_variables
    if not os.getenv(variable_name)
]

if missing_variables:
    raise ValueError(
        "Missing environment variables: " + ", ".join(missing_variables)
    )

print("Environment file:", ENV_PATH)
for variable_name in required_variables:
    print(f"{variable_name} loaded:", bool(os.getenv(variable_name)))


Environment file: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course\.env
GOOGLE_API_KEY loaded: True
PINECONE_API_KEY loaded: True
DOCUMENTATION_INDEX_NAME loaded: True


## 7. Define backend configuration

The embedding settings must exactly match notebook 11.


In [11]:
INDEX_NAME = os.environ["DOCUMENTATION_INDEX_NAME"]
NAMESPACE = "langchain-docs"
EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 1536
CHAT_MODEL = os.getenv("GEMINI_CHAT_MODEL", "gemini-3.5-flash-lite")
RETRIEVAL_K = 4

print("Index name:", INDEX_NAME)
print("Namespace:", NAMESPACE)
print("Embedding model:", EMBEDDING_MODEL)
print("Embedding dimension:", EMBEDDING_DIMENSION)
print("Chat model:", CHAT_MODEL)
print("Retrieval K:", RETRIEVAL_K)


Index name: langchain-docs-helper-gemini
Namespace: langchain-docs
Embedding model: gemini-embedding-001
Embedding dimension: 1536
Chat model: gemini-3.5-flash-lite
Retrieval K: 4


## 8. Validate the Pinecone index and namespace

> This cell calls Pinecone but does not call Gemini.


In [12]:
pinecone_client = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_names = [item["name"] for item in pinecone_client.list_indexes()]
if INDEX_NAME not in index_names:
    raise ValueError(f"Pinecone index '{INDEX_NAME}' does not exist.")

index_description = pinecone_client.describe_index(INDEX_NAME)

if index_description.dimension != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Index dimension is {index_description.dimension}; expected {EMBEDDING_DIMENSION}."
    )

if index_description.metric != "cosine":
    raise ValueError(f"Expected cosine metric; found {index_description.metric}.")

pinecone_index = pinecone_client.Index(host=index_description.host)
index_stats = pinecone_index.describe_index_stats()
namespace_stats = index_stats.namespaces.get(NAMESPACE)
namespace_vector_count = namespace_stats.vector_count if namespace_stats else 0

print("Index ready:", index_description.status["ready"])
print("Index dimension:", index_description.dimension)
print("Index metric:", index_description.metric)
print(f"Vectors in namespace '{NAMESPACE}':", namespace_vector_count)

if namespace_vector_count == 0:
    raise ValueError(
        f"Namespace '{NAMESPACE}' contains no vectors. Complete notebook 11 first."
    )


Index ready: True
Index dimension: 1536
Index metric: cosine
Vectors in namespace 'langchain-docs': 150


## 9. Initialize Gemini embeddings and chat model

Creating these objects makes no model request.


In [13]:
embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    output_dimensionality=EMBEDDING_DIMENSION,
)

llm = ChatGoogleGenerativeAI(
    model=CHAT_MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0,
    max_retries=2,
)

print("Embedding client:", type(embeddings).__name__)
print("Chat client:", type(llm).__name__)


Embedding client: GoogleGenerativeAIEmbeddings
Chat client: ChatGoogleGenerativeAI


## 10. Connect the vector store and create the base retriever

The retriever returns the top four semantically similar documentation chunks.


In [14]:
vector_store = PineconeVectorStore(
    index=pinecone_index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVAL_K},
)

print("Vector store:", type(vector_store).__name__)
print("Retriever:", type(base_retriever).__name__)


Vector store: PineconeVectorStore
Retriever: VectorStoreRetriever


## 11. Understand why a history-aware retriever is needed

A follow-up question can be ambiguous by itself:

```text
User: What is LCEL?
Assistant: ...
User: How do I use it with a retriever?
```

The phrase `it` refers to LCEL. The history-aware retriever first rewrites the follow-up into a standalone search query, such as:

```text
How do I use LCEL with a LangChain retriever?
```

The rewritten query is used only for retrieval. The original conversation remains available to the answer-generation prompt.


## 12. Create the contextualization prompt

The prompt must reformulate the question without answering it.


In [15]:
contextualize_question_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Given the chat history and the latest user question, rewrite the latest "
        "question as a standalone search query that can be understood without the "
        "chat history. Do not answer the question. If the latest question is already "
        "standalone, return it unchanged.",
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

print(contextualize_question_prompt)


input_variables=['chat_history', 'input'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.messag

## 13. Build the history-aware retriever

```text
input + chat_history
        ↓
Gemini reformulates the question when history exists
        ↓
base_retriever searches Pinecone
        ↓
list[Document]
```

With an empty history, the latest question can be passed directly to retrieval.


In [16]:
history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=base_retriever,
    prompt=contextualize_question_prompt,
)

print("History-aware retriever created:", type(history_aware_retriever).__name__)


History-aware retriever created: RunnableBinding


## 14. Create the grounded answer prompt

The prompt explicitly constrains the answer to the retrieved documentation context and requests an honest fallback when evidence is insufficient.


In [17]:
answer_system_message = """You are a LangChain documentation assistant.

Answer the user's question using only the retrieved documentation context below.
If the context does not contain enough evidence, say that the answer is not available
in the retrieved documentation. Do not invent APIs, classes, parameters, or behavior.
Explain code-oriented answers step by step and keep exact class and method names when
they appear in the context.

Retrieved documentation context:
{context}"""

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", answer_system_message),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

print(answer_prompt)


input_variables=['chat_history', 'context', 'input'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_

## 15. Create the document-combination chain

`create_stuff_documents_chain()` inserts all retrieved chunks into the `{context}` placeholder before calling Gemini.

`stuff` means the selected documents are placed together into one LLM prompt. It does not mean that the complete documentation corpus is sent to Gemini.


In [18]:
question_answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=answer_prompt,
)

print("Question-answer document chain created.")


Question-answer document chain created.


## 16. Create the complete conversational retrieval chain

The result preserves both:

- `answer`: Gemini's grounded answer
- `context`: the retrieved LangChain `Document` objects

The frontend needs the context documents to display source links.


In [19]:
rag_chain = create_retrieval_chain(
    retriever=history_aware_retriever,
    combine_docs_chain=question_answer_chain,
)

print("Conversational RAG chain created:", type(rag_chain).__name__)


Conversational RAG chain created: RunnableBinding


## 17. Define source extraction and deduplication

One source page may produce several retrieved chunks. The frontend should show the URL once, in first-retrieved order.


In [20]:
def extract_unique_sources(documents: list[Document]) -> list[str]:
    """Return unique public source URLs while preserving retrieval order."""

    unique_sources = []
    seen_sources = set()

    for document in documents:
        source = str(document.metadata.get("source", "")).strip()

        if not source.startswith(("https://", "http://")):
            continue

        if source not in seen_sources:
            seen_sources.add(source)
            unique_sources.append(source)

    return unique_sources


## 18. Define chat-history normalization

The Streamlit frontend will store simple dictionaries. LangChain expects message objects. This helper accepts either representation.


In [ ]:
def normalize_chat_history(chat_history: Any | None) -> list[BaseMessage]:
    """Convert dictionaries, tuples, or message objects into LangChain messages."""

    if not chat_history:
        return []

    normalized_messages: list[BaseMessage] = []

    for item in chat_history:
        if isinstance(item, BaseMessage):
            normalized_messages.append(item)
            continue

        if isinstance(item, dict):
            role = str(item.get("role", "")).lower()
            content = str(item.get("content", ""))
        elif isinstance(item, (tuple, list)) and len(item) == 2:
            role = str(item[0]).lower()
            content = str(item[1])
        else:
            raise TypeError(
                "Each chat-history item must be a LangChain message, a role/content "
                "dictionary, or a two-item role/content tuple."
            )

        if role in {"user", "human"}:
            normalized_messages.append(HumanMessage(content=content))
        elif role in {"assistant", "ai"}:
            normalized_messages.append(AIMessage(content=content))
        else:
            raise ValueError(f"Unsupported chat-history role: {role}")

    return normalized_messages


## 19. Define the reusable `run_llm()` backend function

This is the primary function that the later Streamlit application will call.

Returned structure:

```python
{
    "answer": "...",
    "context": [Document(...), ...],
    "sources": ["https://...", ...],
    "retrieved_chunk_count": 4,
}
```


In [ ]:
def run_llm(
    query: str,
    chat_history: Any | None = None,
) -> dict[str, Any]:
    """
    Run the conversational documentation RAG backend.

    Parameters
    ----------
    query:
        The latest user question.
    chat_history:
        Optional prior user and assistant messages.

    Returns
    -------
    dict
        Answer text, retrieved context documents, unique source URLs,
        and retrieval metadata needed by the frontend.
    """

    clean_query = query.strip()
    if not clean_query:
        raise ValueError("The user query cannot be empty.")

    normalized_history = normalize_chat_history(chat_history)

    log_info(f"Running documentation RAG for query: {clean_query}")

    result = rag_chain.invoke({
        "input": clean_query,
        "chat_history": normalized_history,
    })

    answer = str(result.get("answer", "")).strip()
    context_documents = result.get("context", [])

    if not isinstance(context_documents, list):
        context_documents = list(context_documents)

    sources = extract_unique_sources(context_documents)

    return {
        "answer": answer,
        "context": context_documents,
        "sources": sources,
        "retrieved_chunk_count": len(context_documents),
        "chat_history_message_count": len(normalized_history),
    }


## 20. Inspect raw retrieval before generating an answer

> This cell makes one Gemini embedding request and one Pinecone query. It does not call the Gemini chat model.

Always inspect retrieval quality before diagnosing answer quality.


In [ ]:
RUN_RETRIEVAL_INSPECTION = False

if RUN_RETRIEVAL_INSPECTION:
    inspection_query = "What is the purpose of RunnablePassthrough?"
    inspection_documents = base_retriever.invoke(inspection_query)

    print("Query:", inspection_query)
    print("Retrieved chunks:", len(inspection_documents))

    for result_number, document in enumerate(inspection_documents, start=1):
        print("=" * 100)
        print("Result:", result_number)
        print("Source:", document.metadata.get("source"))
        print("Source chunk:", document.metadata.get("source_chunk_number"))
        print("Content preview:")
        print(document.page_content[:1400])
        print()
else:
    log_warning("Retrieval inspection skipped. Set RUN_RETRIEVAL_INSPECTION=True to run it.")


## 21. Test a standalone documentation question

> This cell normally makes one query-embedding request, one Pinecone query, and one Gemini answer-generation request.


In [ ]:
RUN_STANDALONE_QUESTION_TEST = False

if RUN_STANDALONE_QUESTION_TEST:
    standalone_result = run_llm(
        "What is RunnablePassthrough and how does assign work?"
    )

    print("Answer:")
    print(standalone_result["answer"])
    print()
    print("Retrieved chunk count:", standalone_result["retrieved_chunk_count"])
    print("Sources:")
    for source in standalone_result["sources"]:
        print("-", source)
else:
    log_warning("Standalone test skipped. Set RUN_STANDALONE_QUESTION_TEST=True to run it.")


## 22. Test conversational follow-up behavior

This test demonstrates why chat history matters.

```text
Turn 1: What is LCEL?
Turn 2: How do I use it with a retriever?
```

For turn 2, the history-aware retriever can resolve `it` to LCEL before searching Pinecone.

> Running both turns makes multiple Gemini and Pinecone requests.


In [ ]:
RUN_CONVERSATIONAL_TEST = False

if RUN_CONVERSATIONAL_TEST:
    conversation_history = []

    first_question = "What is LCEL?"
    first_result = run_llm(first_question, conversation_history)

    conversation_history.extend([
        {"role": "user", "content": first_question},
        {"role": "assistant", "content": first_result["answer"]},
    ])

    second_question = "How do I use it with a retriever?"
    second_result = run_llm(second_question, conversation_history)

    print("First question:", first_question)
    print("First answer:")
    print(first_result["answer"])
    print()
    print("Follow-up question:", second_question)
    print("Follow-up answer:")
    print(second_result["answer"])
    print()
    print("Follow-up sources:")
    for source in second_result["sources"]:
        print("-", source)
else:
    log_warning("Conversational test skipped. Set RUN_CONVERSATIONAL_TEST=True to run it.")


## 23. Test an out-of-scope question

The expected behavior is an evidence-limited response rather than an answer from Gemini's general knowledge.


In [ ]:
RUN_OUT_OF_SCOPE_TEST = False

if RUN_OUT_OF_SCOPE_TEST:
    out_of_scope_result = run_llm(
        "What was the closing stock price of Microsoft yesterday?"
    )

    print("Answer:")
    print(out_of_scope_result["answer"])
    print()
    print("Sources returned:", out_of_scope_result["sources"])
else:
    log_warning("Out-of-scope test skipped. Set RUN_OUT_OF_SCOPE_TEST=True to run it.")


## 24. Optional direct similarity scores

The high-level retriever returns documents without numeric scores. Use this diagnostic only when investigating retrieval ranking.

> This makes a fresh Gemini embedding request and Pinecone query.


In [ ]:
RUN_SCORE_DIAGNOSTIC = False

if RUN_SCORE_DIAGNOSTIC:
    score_query = "How does RunnablePassthrough assign values?"
    scored_results = vector_store.similarity_search_with_score(
        query=score_query,
        k=RETRIEVAL_K,
    )

    for result_number, (document, score) in enumerate(scored_results, start=1):
        print("=" * 100)
        print("Result:", result_number)
        print("Similarity score:", round(score, 4))
        print("Source:", document.metadata.get("source"))
        print(document.page_content[:1000])
        print()
else:
    log_warning("Score diagnostic skipped. Set RUN_SCORE_DIAGNOSTIC=True to run it.")


## 25. Backend error boundary for the frontend

The final Streamlit app should catch exceptions and display a user-friendly message while retaining technical details for logs.


In [ ]:
def run_llm_safely(
    query: str,
    chat_history: Any | None = None,
) -> dict[str, Any]:
    """Call run_llm and return a frontend-safe error structure on failure."""

    try:
        result = run_llm(
            query=query,
            chat_history=chat_history,
        )

        return {
            "ok": True,
            **result,
            "error_type": None,
            "error_message": None,
        }

    except Exception as error:
        log_error(f"Backend failure: {type(error).__name__}: {error}")

        return {
            "ok": False,
            "answer": (
                "The documentation assistant could not complete the request. "
                "Please wait briefly and try again."
            ),
            "context": [],
            "sources": [],
            "retrieved_chunk_count": 0,
            "chat_history_message_count": 0,
            "error_type": type(error).__name__,
            "error_message": str(error),
        }


## 26. Source formatting helper for Streamlit

The frontend can use this helper to turn URLs into numbered Markdown links.


In [ ]:
def format_sources_as_markdown(sources: list[str]) -> str:
    """Format unique source URLs as numbered Markdown links."""

    if not sources:
        return "No source links were returned."

    return "\n".join(
        f"{source_number}. [{source}]({source})"
        for source_number, source in enumerate(sources, start=1)
    )


example_sources = [
    "https://docs.langchain.com/oss/python/langchain/overview",
    "https://docs.langchain.com/oss/python/langchain/retrieval",
]

print(format_sources_as_markdown(example_sources))


## 27. Approximate API calls per user turn

### Standalone question with no chat history

```text
1 Gemini embedding request
1 Pinecone similarity query
1 Gemini answer-generation request
```

### Follow-up question with chat history

```text
1 Gemini reformulation request
1 Gemini embedding request
1 Pinecone similarity query
1 Gemini answer-generation request
```

The exact provider-level request count can vary by integration behavior and retries.

This backend performs no Tavily crawl requests. Tavily was used only during notebook 10.


## 28. What will move into the final application modules

The notebook keeps everything visible for learning. The final Streamlit application will extract reusable pieces into normal Python files:

```text
documentation_helper_backend.py
    - configuration loading
    - Gemini clients
    - Pinecone connection
    - prompts and RAG chain
    - run_llm()
    - run_llm_safely()
    - source helpers

documentation_chatbot_streamlit_gemini.py
    - page configuration
    - session state
    - chat rendering
    - user input
    - spinner and error display
    - source expander
```

This keeps the UI separate from data and model orchestration.


## 29. Handoff to notebook 13

Notebook 13 will prototype the Streamlit-facing behavior:

```text
Session state
+ chat-history messages
+ user input
+ backend result
+ source links
+ clear-chat behavior
```

The final `.py` application will then use the same backend contract:

```python
result = run_llm_safely(
    query=user_question,
    chat_history=st.session_state.messages,
)
```
